In [3]:
import os

In [4]:
%pwd

'c:\\Text-Summarizer-Project\\research'

In [5]:
os.chdir("../")

In [6]:
%pwd

'c:\\Text-Summarizer-Project'

In [7]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_path: Path

In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    eval_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int

In [9]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories


In [10]:
class ConfigurationManager:

    def __init__(
        self,
        config_filepath=config_file_path,
        params_filepath=params_file_path
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


def get_model_trainer_config(self) -> ModelTrainerConfig:

    config = self.config.model_trainer
    params = self.params.TrainingArguments

    create_directories([config.root_dir])

    model_trainer_config = ModelTrainerConfig(
        root_dir=Path(config.root_dir),
        data_path=Path(config.data_path),
        model_ckpt=config.model_ckpt,

        num_train_epochs=params.num_train_epochs,
        warmup_steps=params.warmup_steps,
        per_device_train_batch_size=params.per_device_train_batch_size,
        weight_decay=params.weight_decay,
        logging_steps=params.logging_steps,
        evaluation_strategy=params.evaluation_strategy,
        eval_steps=params.eval_steps,
        save_steps=params.save_steps,
        gradient_accumulation_steps=params.gradient_accumulation_steps
    )

    return model_trainer_config

In [11]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch

In [12]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"

        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)

        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_ckpt
        ).to(device)

        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir,
            num_train_epochs=self.config.num_train_epochs,
            warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            weight_decay=self.config.weight_decay,
            logging_steps=self.config.logging_steps,
            evaluation_strategy=self.config.evaluation_strategy,
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            report_to="none",
            fp16=torch.cuda.is_available()
        )

        # Small subset for faster testing
        small_train = dataset_samsum_pt["train"].shuffle(seed=42).select(range(100))
        small_val = dataset_samsum_pt["validation"].shuffle(seed=42).select(range(20))

        trainer = Trainer(
            model=model_pegasus,
            args=trainer_args,
            train_dataset=small_train,
            eval_dataset=small_val
        )

        trainer.train()

        model_pegasus.save_pretrained(
            os.path.join(self.config.root_dir, "pegasus-samsum-model")
        )

        tokenizer.save_pretrained(
            os.path.join(self.config.root_dir, "tokenizer")
        )

In [13]:
import os

print("Current directory:", os.getcwd())
print("Contents of current directory:")
print(os.listdir())

print("\nContents of parent directory:")
print(os.listdir(".."))

Current directory: c:\Text-Summarizer-Project
Contents of current directory:
['.git', '.github', '.gitignore', '.vscode', 'app.py', 'artifacts', 'config', 'data.zip', 'Dockerfile', 'LICENSE', 'logs', 'main.py', 'params.yaml', 'README.md', 'requirements.txt', 'research', 'setup.py', 'src', 'templatemain.py']

Contents of parent directory:
['$Recycle.Bin', '$WINDOWS.~BT', '.GamingRoot', 'Autodesk', 'Config.Msi', 'Documents and Settings', 'Drivers', 'DumpStack.log', 'DumpStack.log.tmp', 'Game', 'hiberfil.sys', 'id card', 'inetpub', 'Intel', 'jupyter', 'MinGW', 'msdia80.dll', 'msys64', 'OneDriveTemp', 'oraclexe', 'pagefile.sys', 'PerfLogs', 'Program Files', 'Program Files (x86)', 'ProgramData', 'Python314', 'pythonbeginning', 'Recovery', 'swapfile.sys', 'System Volume Information', 'Text-Summarizer-Project', 'Text-Summarizer1', 'TurboC++', 'Users', 'Windows', 'XboxGames']


In [16]:
from textsummarizer.config.configuration import ConfigurationManager
from textsummarizer.components.model_trainer import ModelTrainer

In [17]:
import os
os.chdir("C:/Text-Summarizer-Project")
print(os.getcwd())

C:\Text-Summarizer-Project


In [18]:
config = ConfigurationManager()

[2026-06-18 23:49:05,982: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-06-18 23:49:05,982: INFO: common]: yaml file: params.yaml loaded successfully
[2026-06-18 23:49:05,989: INFO: common]: created directory at: artifacts


In [19]:
from textsummarizer.utils.common import read_yaml
from textsummarizer.constants import *

config = read_yaml(config_file_path)

print(config)
print(config.model_trainer)

[2026-06-18 23:49:07,241: INFO: common]: yaml file: config\config.yaml loaded successfully
{'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'source_URL': 'https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_dir': 'artifacts/data_ingestion'}, 'data_validation': {'root_dir': 'artifacts/data_validation', 'status_file': 'artifacts/data_validation/status.txt', 'all_required_files': ['train', 'test', 'validation']}, 'data_transformation': {'root_dir': 'artifacts/data_transformation', 'data_path': 'artifacts/data_ingestion/samsum_dataset', 'tokenizer_name': 'google/pegasus-cnn_dailymail'}, 'model_trainer': {'root_dir': 'artifacts/model_trainer', 'data_path': 'artifacts/data_transformation/samsum_dataset', 'model_ckpt': 'google/pegasus-cnn_dailymail'}}
{'root_dir': 'artifacts/model_trainer', 'data_path': 'artifacts/data_transformation/samsum_dataset', 'model_ckpt

In [20]:
from textsummarizer.config.configuration import ConfigurationManager

config = ConfigurationManager()
model_trainer_config = config.get_model_trainer_config()

print(model_trainer_config)

[2026-06-18 23:49:08,460: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-06-18 23:49:08,469: INFO: common]: yaml file: params.yaml loaded successfully
[2026-06-18 23:49:08,471: INFO: common]: created directory at: artifacts
[2026-06-18 23:49:08,472: INFO: common]: created directory at: artifacts/model_trainer
ModelTrainerConfig(root_dir=WindowsPath('artifacts/model_trainer'), data_path=WindowsPath('artifacts/data_transformation/samsum_dataset'), model_ckpt='google/pegasus-cnn_dailymail', num_train_epochs=1, warmup_steps=100, per_device_train_batch_size=1, weight_decay=0.01, logging_steps=50, evaluation_strategy='steps', eval_steps=1000, save_steps=1000, gradient_accumulation_steps=16)


In [21]:
import os
import sys

os.chdir("C:/Text-Summarizer-Project")
sys.path.append("C:/Text-Summarizer-Project/src")

from textsummarizer.config.configuration import ConfigurationManager
from textsummarizer.components.model_trainer import ModelTrainer

In [22]:
config = ConfigurationManager()
model_trainer_config = config.get_model_trainer_config()

print(model_trainer_config)

[2026-06-18 23:49:10,840: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-06-18 23:49:10,852: INFO: common]: yaml file: params.yaml loaded successfully
[2026-06-18 23:49:10,854: INFO: common]: created directory at: artifacts
[2026-06-18 23:49:10,856: INFO: common]: created directory at: artifacts/model_trainer
ModelTrainerConfig(root_dir=WindowsPath('artifacts/model_trainer'), data_path=WindowsPath('artifacts/data_transformation/samsum_dataset'), model_ckpt='google/pegasus-cnn_dailymail', num_train_epochs=1, warmup_steps=100, per_device_train_batch_size=1, weight_decay=0.01, logging_steps=50, evaluation_strategy='steps', eval_steps=1000, save_steps=1000, gradient_accumulation_steps=16)


In [ ]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2026-06-18 23:49:12,425: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-06-18 23:49:12,428: INFO: common]: yaml file: params.yaml loaded successfully
[2026-06-18 23:49:12,430: INFO: common]: created directory at: artifacts
[2026-06-18 23:49:12,432: INFO: common]: created directory at: artifacts/model_trainer
